In [1]:
###!/usr/bin/env python
################################################
# New style 
# ###############################################
import sys

rootdir_ = '../'
if ( rootdir_ not in sys.path ):
    sys.path.append(rootdir_)
    print( f" a path to {rootdir_} added in {__name__} ")


from Utils import GridUtils as GrU
from Utils import MakePressures as MkP
from Utils import utils as uti
from Utils import MyConstants as Co
from Utils import time_utils as tuti
from Utils import numerical_utils as nuti

import analysis_utils as auti
import file_utils as futi
import event_utils as euti
import event_io as eio


#from PyRegridding.Utils import MakePressures as MkP
#from Drivers import RegridField as RgF
import RegridField as RgF

# The usual
from datetime import date
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# for smoothing , nonlienar colors ...
from scipy.ndimage import uniform_filter
from scipy.ndimage import gaussian_filter
import matplotlib.colors as mcolors

# Some other useful packages 
import copy
import time
import cftime
import yaml
import numbers
import pickle

# Some other useful packages 
import importlib
from pathlib import Path

import mlp_to_pptx


importlib.reload( auti )
importlib.reload( futi )
importlib.reload( euti )
importlib.reload( eio )

Rdair=Co.Rdair()


 a path to ../ added in __main__ 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
Using Flexible parallel/serial VertRegrid 
 Utils.MyConstants in /glade/work/juliob/HiRes_ana_dev/Drivers/Utils 
 a path to /glade/work/juliob added in Utils.numerical_utils 


In [2]:
# This allow both dict.key and dict['key'] syntax
class AttrDict(dict):
    def __getattr__(self, key):
        try:
            return self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")

    def __setattr__(self, key, value):
        self[key] = value

    def __delattr__(self, key):
        try:
            del self[key]
        except KeyError:
            raise AttributeError(f"'AttrDict' object has no attribute '{key}'")



In [3]:
%%time
original_3x3_study=False
current_study=True

### Initialize to False
subsample_time_12_06_00 = False
transfer_x_ne240 = False
transfer_x_mpas  = False



bdir = f"/glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL"

fEl1 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl"
fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl"



#fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl"

#fEl1 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl"
#fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl"

#fEl1 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_Frac100%_v4.pkl"
#fEl2 = f"{bdir}/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_35N-70N_ocean_EvZ10km_rho_epwp_subt_ftp5X5_Frac100%_v4.pkl"


print( f"Reading in from \n {fEl1} ")
with open(f'{fEl1}', 'rb') as f:
    El = pickle.load(f)

print( f"Reading in from \n {fEl2} ")
with open(f'{fEl2}', 'rb') as f:
    El2 = pickle.load(f)


Reading in from 
 /glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_ftp5X5_v2.pkl 
Reading in from 
 /glade/derecho/scratch/juliob/archive/GW_event_analysis/PKL/cam77_dyamond1_prod1_2016-08-01-10800-x-2016-08-31-75600_60S-40S_ocean_EvZ10km_rho_epwp_subt_ftp5X5_v3.pkl 
CPU times: user 122 ms, sys: 56 s, total: 56.1 s
Wall time: 15min 52s


In [9]:
Eco=El[0]
Eco2=El2[0]


In [11]:
print(Eco.u_4D.shape)
print(Eco2.u_4D.shape)


(18309, 4, 58, 11, 11)
(18309, 3, 58, 11, 11)
